# Sistema Híbrido de Recomendación de Libros

## Motor de recomendación basado en contenido y filtrado colaborativo utilizando datos de Goodreads

### Descripción

Los sistemas de recomendación desempeñan un papel fundamental en plataformas digitales con grandes catálogos de contenido, permitiendo personalizar la experiencia de los usuarios y mejorar el descubrimiento de nuevos productos.

En este proyecto se desarrolla un sistema híbrido de recomendación de libros utilizando información procedente de Goodreads. La solución combina técnicas de Procesamiento de Lenguaje Natural (NLP), representación vectorial mediante TF-IDF y modelos de filtrado colaborativo para identificar libros con características similares y generar recomendaciones personalizadas.

El desarrollo sigue un flujo de trabajo completo de Ciencia de Datos, desde la preparación de los datos hasta la construcción y evaluación del sistema de recomendación.

## Objetivos del proyecto

El objetivo principal consiste en diseñar un sistema de recomendación capaz de sugerir libros relevantes a partir de dos enfoques complementarios:

- **Recomendación basada en contenido**, utilizando las etiquetas asociadas a cada libro.
- **Filtrado colaborativo**, aprovechando las valoraciones históricas de los usuarios.

Para ello se desarrollará un pipeline que incluye:

- Preparación y validación de datos.
- Ingeniería de características textuales.
- Procesamiento mediante técnicas de NLP.
- Representación vectorial utilizando TF-IDF.
- Cálculo de similitud entre libros.
- Modelado mediante SVD y LDA.
- Construcción de un sistema híbrido de recomendación.

## Tecnologías utilizadas

| Área | Herramientas |
|------|--------------|
| Lenguaje | Python |
| Manipulación de datos | pandas, NumPy |
| Visualización | Matplotlib |
| NLP | NLTK |
| Machine Learning | scikit-learn |
| Recomendación | TF-IDF, Cosine Similarity, SVD |
| Modelado temático | LDA |

## Flujo de trabajo

El desarrollo del proyecto seguirá las siguientes etapas:

1. Configuración del entorno.
2. Carga y validación de los datos.
3. Ingeniería de variables.
4. Preparación del corpus textual.
5. Vectorización mediante TF-IDF.
6. Construcción del recomendador basado en contenido.
7. Modelado mediante filtrado colaborativo.
8. Integración del sistema híbrido.
9. Análisis de resultados y conclusiones.

# 1. Configuración del entorno

En esta sección se importan las librerías necesarias y se configura el entorno de trabajo para garantizar la reproducibilidad del análisis.

In [1]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from IPython.display import display

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.3f}".format)

# 2. Localización de los datos

El proyecto utiliza el conjunto de datos público de Goodreads. Antes de iniciar el procesamiento se verifica la existencia de los archivos necesarios para garantizar una ejecución reproducible.

In [4]:
DATA_DIR = Path("data")

required_files = [
    "books.csv",
    "book_tags.csv",
    "tags.csv",
    "ratings.csv"
]

for filename in required_files:
    filepath = DATA_DIR / filename

    if filepath.exists():
        print(f"✓ {filename}")
    else:
        print(f"✗ {filename} no encontrado")

✓ books.csv
✓ book_tags.csv
✓ tags.csv
✓ ratings.csv


## Estructura del conjunto de datos

El sistema de recomendación utiliza diferentes fuentes de información:

- **books.csv**: metadatos de los libros.
- **book_tags.csv**: relación entre libros y etiquetas.
- **tags.csv**: catálogo de etiquetas.
- **ratings.csv**: valoraciones realizadas por los usuarios.

La combinación de estas fuentes permite construir tanto un recomendador basado en contenido como un modelo de filtrado colaborativo.

In [5]:
books[[
    "title",
    "authors",
    "average_rating",
    "ratings_count"
]].head()

,title,authors,average_rating,ratings_count
book_id,,,,
0,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,4.34,4780653
1,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",4.44,4602479
2,"Twilight (Twilight, #1)",Stephenie Meyer,3.57,3866839
3,To Kill a Mockingbird,Harper Lee,4.25,3198671
4,The Great Gatsby,F. Scott Fitzgerald,3.89,2683664


# 3. Ingeniería de características textuales

El dataset incorpora información semántica generada por la comunidad de Goodreads mediante etiquetas asociadas a cada libro.

Estas etiquetas describen géneros, temáticas, estilos narrativos y otros atributos relevantes.

A partir de ellas se construirá la representación textual utilizada por los modelos de recomendación.

## Carga de etiquetas

Las etiquetas aportadas por los usuarios contienen información semántica relevante sobre géneros, temáticas y características de los libros.

In [6]:
tags = pd.read_csv("./data/tags.csv")
book_tags = pd.read_csv("./data/book_tags.csv")

In [7]:
print(f"Tags disponibles: {tags.shape[0]:,}")
print(f"Relaciones libro-tag: {book_tags.shape[0]:,}")

Tags disponibles: 34,252
Relaciones libro-tag: 999,912


## Integración de etiquetas

Se sustituyen los identificadores numéricos por sus etiquetas descriptivas para facilitar el procesamiento posterior.

In [8]:
book_tags = (
    book_tags
    .merge(tags, on="tag_id")
    .drop(columns=["count", "tag_id"])
)

book_tags.head()

,gr_book_id,tag_name
0,1,to-read
1,1,fantasy
2,1,favorites
3,1,currently-reading
4,1,young-adult


## Filtrado de etiquetas poco representativas

Las etiquetas extremadamente poco frecuentes suelen aportar ruido al modelo y dificultan la generalización.

Se conservarán únicamente aquellas que aparecen al menos tres veces en el catálogo.

In [9]:
book_tags = (
    book_tags
    .groupby("tag_name")
    .filter(lambda x: len(x) >= 3)
)

## Eliminación de etiquetas genéricas

Algunas etiquetas reflejan acciones de los usuarios y no describen realmente el contenido de los libros.

Estas etiquetas se eliminan para mejorar la calidad de la representación textual.

In [10]:
target_tags = [
    "read",
    "favo",
    "own",
    "top",
    "book",
    "librar"
]

book_tags = book_tags[
    ~book_tags["tag_name"].str.contains(
        "|".join(target_tags),
        case=False,
        na=False
    )
]

## Construcción del corpus de etiquetas

Las etiquetas de cada libro se agregan en un único campo textual para facilitar su utilización en las etapas posteriores de NLP.

In [11]:
df_book_tag_text = (
    book_tags
    .groupby("gr_book_id")["tag_name"]
    .apply(" ".join)
    .reset_index()
    .rename(columns={"tag_name": "text_tags"})
)

df_book_tag_text.head()

,gr_book_id,text_tags
0,1,fantasy young-adult fiction harry-potter ya se...
1,2,fantasy children children-s default fiction yo...
2,3,fantasy young-adult fiction harry-potter ya se...
3,5,fantasy young-adult fiction harry-potter ya se...
4,6,fantasy young-adult fiction harry-potter ya se...


## Integración de etiquetas en el dataset principal

Se incorpora la información textual derivada de las etiquetas a cada libro.

In [12]:
books = books.merge(
    df_book_tag_text,
    on="gr_book_id",
    how="left"
)

books["text_tags"] = books["text_tags"].fillna("")

# 4. Construcción de la representación textual

Las etiquetas asociadas a cada libro constituyen la principal fuente de información semántica disponible.

Para cada título se genera una representación textual consolidada que servirá posteriormente como entrada para los procesos de:

- Limpieza y normalización del texto
- Vectorización TF-IDF
- Cálculo de similitud
- Recomendación basada en contenido
- Modelado temático mediante LDA

In [13]:
books["text"] = books["text_tags"]

In [14]:
books = books[
    books["text"].str.len() > 0
].copy()

print(f"Libros disponibles: {len(books):,}")

Libros disponibles: 10,000


In [15]:
books[[
    "title",
    "authors",
    "text"
]].head()

,title,authors,text
0,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,young-adult fiction fantasy ya science-fiction...
1,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",fantasy young-adult fiction harry-potter ya se...
2,"Twilight (Twilight, #1)",Stephenie Meyer,young-adult fantasy vampires ya fiction parano...
3,To Kill a Mockingbird,Harper Lee,classics classic historical-fiction school clà...
4,The Great Gatsby,F. Scott Fitzgerald,classics fiction classic literature school his...


## Resultado

Tras el proceso de preparación se dispone de una representación textual para cada libro basada en conocimiento colectivo generado por los usuarios de Goodreads.

Esta representación será utilizada en las siguientes etapas para construir modelos de recomendación basados en similitud semántica.

## 5. Preprocesamiento del texto

Antes de construir la representación vectorial de los libros es necesario normalizar el texto para reducir ruido y mejorar la calidad de los modelos.

En esta etapa se aplicarán las siguientes técnicas de Procesamiento de Lenguaje Natural (NLP):

- Tokenización
- Eliminación de *stop words*
- Lematización
- Eliminación de entidades nombradas

Estas transformaciones permiten obtener representaciones más compactas y comparables entre documentos.

In [16]:
import nltk

from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

## Descarga de recursos de NLTK

La primera ejecución requiere descargar los recursos lingüísticos utilizados durante el preprocesamiento.

In [17]:
resources = [
    "punkt",
    "punkt_tab",
    "stopwords",
    "wordnet"
]

for resource in resources:
    nltk.download(resource)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LUIS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\LUIS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LUIS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\LUIS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Inicialización de componentes NLP

Se configuran los objetos encargados de realizar la tokenización, eliminación de palabras vacías y lematización del texto.

In [18]:
tokenizer = RegexpTokenizer(r"[a-zA-Z]+")

stopwords_en = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

## Función de limpieza del texto

Se define un pipeline de limpieza que integra todas las operaciones anteriores y devuelve una representación normalizada de cada documento.

In [19]:
def clean_text(text):
    """
    Normaliza el texto eliminando palabras vacías y aplicando lematización.
    """

    tokens = tokenizer.tokenize(text.lower())

    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stopwords_en
    ]

    return " ".join(tokens)

## Ejemplo de transformación

Antes de procesar el corpus completo resulta útil comprobar visualmente el efecto del pipeline de limpieza sobre un documento.

In [20]:
sample = books.iloc[0]["text"]

print("Texto original:\n")
print(sample)

print("\n" + "-" * 80 + "\n")

print("Texto procesado:\n")
print(clean_text(sample))

Texto original:

young-adult fiction fantasy ya science-fiction sci-fi series romance adventure hunger-games kindle teen post-apocalyptic the-hunger-games action suzanne-collins ya-fiction survival sci-fi-fantasy scifi 5-stars novels young-adult-fiction thriller futuristic default fantasy-sci-fi suspense faves audio teen-fiction ya-lit finished action-adventure love love-triangle favs loved coming-of-age borrowed english to-buy novel scifi-fantasy ya-fantasy future contemporary trilogy reviewed speculative-fiction finished-series sf completed-series drama 5-star

--------------------------------------------------------------------------------

Texto procesado:

young adult fiction fantasy ya science fiction sci fi series romance adventure hunger game kindle teen post apocalyptic hunger game action suzanne collins ya fiction survival sci fi fantasy scifi star novel young adult fiction thriller futuristic default fantasy sci fi suspense faves audio teen fiction ya lit finished action adv

## 6. Sistema de recomendación basado en contenido

Una vez obtenida la representación vectorial de cada libro, el siguiente paso consiste en medir el grado de similitud existente entre ellos.

Para ello se utiliza la **similitud del coseno**, una métrica ampliamente empleada en sistemas de recomendación y recuperación de información.

Dos libros serán considerados similares cuando sus representaciones TF-IDF presenten una orientación próxima en el espacio vectorial.

In [21]:
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
book_vectors

NameError: name 'book_vectors' is not defined

In [22]:
similarity_matrix = cosine_similarity(book_vectors)

print(
    f"Matriz de similitud creada con dimensiones "
    f"{similarity_matrix.shape}"
)

NameError: name 'book_vectors' is not defined

## Índice de búsqueda

Para agilizar las consultas se construye un índice que relaciona cada título con su posición dentro de la matriz de similitud.

In [ ]:
book_index = pd.Series(
    books.index,
    index=books["title"]
).drop_duplicates()

book_index.head()

## Función de recomendación

Se implementa una función reutilizable que devuelve los libros más similares a partir de un título de referencia.

Esta función constituye el núcleo del sistema de recomendación basado en contenido.

In [ ]:
def recommend_books(title, top_n=10):

    if title not in book_index:
        raise ValueError(f'"{title}" no existe en el catálogo.')

    idx = book_index[title]

    similarity_scores = list(
        enumerate(similarity_matrix[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    recommendations = books.iloc[
        [i[0] for i in similarity_scores]
    ][[
        "title",
        "authors",
        "average_rating"
    ]].copy()

    recommendations["similarity"] = [
        score
        for _, score in similarity_scores
    ]

    return recommendations.reset_index(drop=True)

## Ejemplo de recomendación

Como validación del modelo se generan recomendaciones para un título presente en el catálogo.

In [ ]:
books["title"].sample(10, random_state=42)

In [ ]:
recommend_books(
    "The Hunger Games",
    top_n=10
)

## Interpretación de resultados

Las recomendaciones obtenidas presentan una elevada similitud temática al compartir etiquetas representativas del contenido de los libros.

Este enfoque permite construir un sistema de recomendación incluso cuando no se dispone de información histórica sobre las preferencias de los usuarios.

## 7. Análisis exploratorio del espacio vectorial

Aunque las representaciones TF-IDF contienen miles de dimensiones, resulta útil proyectarlas sobre un espacio bidimensional para explorar visualmente la distribución de los libros.

In [ ]:
from sklearn.decomposition import TruncatedSVD

In [ ]:
svd = TruncatedSVD(
    n_components=2,
    random_state=42
)

projection = svd.fit_transform(book_vectors)

## Visualización del espacio semántico

Cada punto representa un libro.

La proximidad entre puntos indica una mayor similitud en términos de las etiquetas utilizadas para describirlos.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,8))

plt.scatter(
    projection[:,0],
    projection[:,1],
    s=10,
    alpha=0.6
)

plt.title("Representación bidimensional del catálogo")

plt.xlabel("Componente 1")

plt.ylabel("Componente 2")

plt.show()

## 8. Descubrimiento de temáticas mediante LDA

Además de recomendar libros similares, resulta interesante identificar automáticamente las principales temáticas presentes en el catálogo.

Para ello se emplea **Latent Dirichlet Allocation (LDA)**, un modelo probabilístico capaz de descubrir tópicos latentes a partir del corpus textual.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
count_vectorizer = CountVectorizer(
    max_features=5000
)

count_matrix = count_vectorizer.fit_transform(clean_books)

In [ ]:
lda = LatentDirichletAllocation(
    n_components=10,
    random_state=42
)

lda.fit(count_matrix)

## Palabras más representativas por tópico

A continuación se muestran los términos con mayor peso dentro de cada uno de los tópicos identificados.

In [ ]:
feature_names = count_vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):

    print(f"\nTópico {topic_idx + 1}")

    print("-" * 40)

    words = [
        feature_names[i]
        for i in topic.argsort()[-10:][::-1]
    ]

    print(", ".join(words))

# Conclusiones

En este proyecto se ha desarrollado un sistema de recomendación basado en contenido utilizando información semántica generada por la comunidad de Goodreads.

El flujo de trabajo ha incluido:

- Preparación y limpieza del corpus textual.
- Representación vectorial mediante TF-IDF.
- Cálculo de similitud entre libros utilizando similitud del coseno.
- Construcción de un motor de recomendación reutilizable.
- Exploración de la estructura temática del catálogo mediante LDA.

Este tipo de sistemas constituye una base sólida para el desarrollo de soluciones híbridas capaces de combinar información basada en contenido con técnicas de filtrado colaborativo y modelos de aprendizaje profundo.